<a href="https://colab.research.google.com/github/vivaanjain20-ctrl/CodingClubMLRecruitmentTask/blob/main/Model_Building_and_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Feature Engineering for training dataset


In [26]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Load data
df = pd.read_csv('/content/tour_logs_cleaned_final.csv')

# --- Feature Engineering ---
df['Is_Weekend'] = df['Day_Name'].isin(['Saturday', 'Sunday']).astype(int)

# Volume Impact (Alpha & Delta)
df['Positive_Volume_Impact'] = df['Volume_Level'] * df['Venue_ID'].isin(['V_Alpha', 'V_Delta']).astype(int)

# Beta Features
df['Beta_Night_Bonus'] = ((df['Venue_ID'] == 'V_Beta') & (df['Hour'] >= 18)).astype(int)
df['Beta_Denim_Bonus'] = ((df['Venue_ID'] == 'V_Beta') & (df['Band_Outfit'] == 'Denim')).astype(int)
df['Price_Beta'] = df['Ticket_Price'] * (df['Venue_ID'] == 'V_Beta').astype(int)
df['Storm_Penalty_Beta'] = ((df['Venue_ID'] == 'V_Beta') & (df['Weather'] == 'Stormy')).astype(int)

# Gamma Features
df['Gamma_Price_Impact'] = df['Ticket_Price'] * (df['Venue_ID'] == 'V_Gamma').astype(int)
df['Gamma_Volume_Penalty'] = df['Volume_Level'] * (df['Venue_ID'] == 'V_Gamma').astype(int)
df['Gamma_Leather_Bonus'] = ((df['Venue_ID'] == 'V_Gamma') & (df['Band_Outfit'] == 'Leather')).astype(int)

# Delta Features
df['Crowd_Delta'] = df['Crowd_Size'] * (df['Venue_ID'] == 'V_Delta').astype(int)
df['Delta_Spandex_Bonus'] = ((df['Venue_ID'] == 'V_Delta') & (df['Band_Outfit'] == 'Spandex')).astype(int)
df['Storm_Penalty_Delta'] = ((df['Venue_ID'] == 'V_Delta') & (df['Weather'] == 'Stormy')).astype(int)

# --- Drop Columns (Clean Version) ---
# I put this on one line to prevent indentation errors
to_drop = ['Gig_ID', 'Merch_Sales_Post_Show', 'Date', 'Time', 'Day_Name', 'Hour', 'Time_of_Day', 'Day_of_Week', 'Storm_Penalty_Beta', 'Storm_Penalty_Delta']

X = df.drop(columns=to_drop, errors='ignore')

# Target
y = X.pop('Crowd_Energy')

# Encoding & Split
X = pd.get_dummies(X, drop_first=True)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Training
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

# Evaluation
val_rmse = np.sqrt(mean_squared_error(y_val, rf.predict(X_val)))
print(f"Validation RMSE: {val_rmse:.4f}")

Validation RMSE: 12.2075


#Training

In [27]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5, 10]
}

rf_base = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    verbose=1,
    n_jobs=-1
)

print("Starting Hyperparameter Tuning...")
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
best_rmse = -grid_search.best_score_

print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best CV RMSE: {best_rmse:.4f}")

val_preds_tuned = best_rf.predict(X_val)
val_rmse_tuned = np.sqrt(mean_squared_error(y_val, val_preds_tuned))
print(f"Validation RMSE after Tuning: {val_rmse_tuned:.4f}")

Starting Hyperparameter Tuning...
Fitting 3 folds for each of 18 candidates, totalling 54 fits

Best Parameters: {'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 200}
Best CV RMSE: 13.1048
Validation RMSE after Tuning: 12.1618


# Cleaning and Prepping Test Data

In [28]:
import pandas as pd
import numpy as np

df_test = pd.read_csv("/content/tour_logs_test_input.csv")

submission_ids = df_test['Gig_ID'].copy()

df_test['Ticket_Price'] = df_test['Ticket_Price'].astype(str).str.replace('Free', '0', case=False)

df_test['Currency'] = df_test['Ticket_Price'].str.extract(r'([$£€A-Z]+)')
df_test['Currency'] = df_test['Currency'].fillna('$').str.replace('USD', '$')
df_test['Ticket_Price'] = df_test['Ticket_Price'].str.replace(r'[^\d.]', '', regex=True)
df_test['Ticket_Price'] = pd.to_numeric(df_test['Ticket_Price'], errors='coerce')

rates = {'£': 1.27, '€': 1.09, '$': 1.0}
df_test['Rate'] = df_test['Currency'].map(rates).fillna(1.0)
df_test['Ticket_Price'] = df_test['Ticket_Price'] * df_test['Rate']
df_test.drop(columns=['Rate', 'Currency'], inplace=True)

train_median_price = 55.0
df_test['Ticket_Price'] = df_test['Ticket_Price'].fillna(train_median_price)

df_test['clean_datetime'] = pd.to_datetime(df_test['Show_DateTime'], format='mixed', dayfirst=True, errors='coerce')

df_test['Hour'] = df_test['clean_datetime'].dt.hour
df_test['Hour'] = df_test['Hour'].fillna(20)

df_test['Day_Name'] = df_test['clean_datetime'].dt.day_name()
df_test['Day_Name'] = df_test['Day_Name'].fillna('Saturday')

cols = ['Crowd_Size', 'Volume_Level']
for col in cols:
    df_test[col] = pd.to_numeric(df_test[col], errors='coerce')
    if col == 'Crowd_Size':
        df_test[col] = df_test[col].fillna(500)
    elif col == 'Volume_Level':
        df_test[col] = df_test[col].fillna(4)

df_test['Is_Weekend'] = df_test['Day_Name'].isin(['Saturday', 'Sunday']).astype(int)

df_test['Positive_Volume_Impact'] = df_test['Volume_Level'] * df_test['Venue_ID'].isin(['V_Alpha', 'V_Delta']).astype(int)
df_test['Beta_Night_Bonus'] = ((df_test['Venue_ID'] == 'V_Beta') & (df_test['Hour'] >= 18)).astype(int)
df_test['Beta_Denim_Bonus'] = ((df_test['Venue_ID'] == 'V_Beta') & (df_test['Band_Outfit'] == 'Denim')).astype(int)
df_test['Price_Beta'] = df_test['Ticket_Price'] * (df_test['Venue_ID'] == 'V_Beta').astype(int)
df_test['Storm_Penalty_Beta'] = ((df_test['Venue_ID'] == 'V_Beta') & (df_test['Weather'] == 'Stormy')).astype(int)
df_test['Gamma_Price_Impact'] = df_test['Ticket_Price'] * (df_test['Venue_ID'] == 'V_Gamma').astype(int)
df_test['Gamma_Volume_Penalty'] = df_test['Volume_Level'] * (df_test['Venue_ID'] == 'V_Gamma').astype(int)
df_test['Gamma_Leather_Bonus'] = ((df_test['Venue_ID'] == 'V_Gamma') & (df_test['Band_Outfit'] == 'Leather')).astype(int)
df_test['Crowd_Delta'] = df_test['Crowd_Size'] * (df_test['Venue_ID'] == 'V_Delta').astype(int)
df_test['Delta_Spandex_Bonus'] = ((df_test['Venue_ID'] == 'V_Delta') & (df_test['Band_Outfit'] == 'Spandex')).astype(int)
df_test['Storm_Penalty_Delta'] = ((df_test['Venue_ID'] == 'V_Delta') & (df_test['Weather'] == 'Stormy')).astype(int)

cols_to_drop = ['Gig_ID', 'Merch_Sales_Post_Show', 'Date', 'Time', 'clean_datetime', 'Show_DateTime', 'Day_of_Week', 'Day_Name', 'Hour', 'Crowd_Energy']
X_test = df_test.drop(columns=[c for c in cols_to_drop if c in df_test.columns], errors='ignore')

X_test = pd.get_dummies(X_test, columns=['Venue_ID', 'Weather', 'Moon_Phase', 'Band_Outfit'], drop_first=True)

print("Test Data Ready for Prediction.")
print(f"Rows: {len(X_test)}")

Test Data Ready for Prediction.
Rows: 500


# Making Predictions

In [29]:
train_cols = X_train.columns

X_test_aligned = X_test.reindex(columns=train_cols, fill_value=0)

print("Columns aligned. Shape:", X_test_aligned.shape)

final_predictions = rf.predict(X_test_aligned)
submission = pd.DataFrame({
    'Gig_ID': submission_ids,
    'Crowd_Energy': final_predictions
})

submission.to_csv('submission.csv', index=False)
print("SUCCESS! 'submission.csv' generated.")
print(submission.head())

Columns aligned. Shape: (500, 29)
SUCCESS! 'submission.csv' generated.
     Gig_ID  Crowd_Energy
0  Gig_0000     61.477895
1  Gig_0001     65.403491
2  Gig_0002     40.040999
3  Gig_0003      3.944374
4  Gig_0004     46.962981
